# 1. Set up enviroment and data

## 1.1. Import libraries

In [73]:
import pandas as pd
import datetime as dt
import numpy as np
import seaborn as sns

## 1.2. Import data to dataframes

In [74]:
data1=pd.read_csv('./data/CusData.csv')
data1.drop(columns=["Unnamed: 0"], inplace=True)
data1.head()

In [75]:
data2=pd.read_csv('./data/TransactionData.csv')
data2.head()

In [76]:
data=pd.DataFrame(data2)

In [77]:
data.drop(columns=["Unnamed: 0",'FOB_RACING_TURNOVER',	'FOB_SPORT_TURNOVER', 'PARI_RACING_TURNOVER', 'PARI_SPORT_TURNOVER', 'DIVIDENDS_PAID',	'GROSS_MARGIN',	'TICKETS'], inplace=True)
data.head()

# 2. Divide data by year

In [78]:
# Chia DataFrame thành hai DataFrame riêng biệt cho năm 2021 và 2022
df_2021 = data[data['YEAR'] == 2021]
df_2022 = data[data['YEAR'] == 2022]

# 3. RFM Calculation

In [79]:
# Tìm ngày lớn nhất
df_2021['DATE_DIM'] = pd.to_datetime(df_2021['DATE_DIM'])
max_date = df_2021['DATE_DIM'].max()
current_date = max_date + pd.Timedelta(days=1)
print("Ngày hiện tại:", current_date)

In [80]:
# Tính toán Recency bằng cách tính số ngày kể từ ngày gần nhất
df_2021['recency'] = (current_date - df_2021['DATE_DIM']).dt.days
df_2021.head()

In [81]:
# Tính toán Monetary
df_2021['money'] = df_2021['TOTAL_TURNOVER']
df_2021.head()

In [82]:
# Tính toán Frequency bằng cách đếm số lần xuất hiện của mỗi BET_ACCOUNT_NUM_HASH
freq_data = df_2021.groupby('BET_ACCOUNT_NUM_HASH').size().reset_index(name='freq')

In [83]:
# Kết hợp dữ liệu recency, monetary và frequency lại với nhau
rfm_df_2021 = df_2021.groupby('BET_ACCOUNT_NUM_HASH').agg({'recency': 'min', 'money': 'sum'}).reset_index()
rfm_df_2021 = pd.merge(rfm_df_2021, freq_data, on='BET_ACCOUNT_NUM_HASH')
rfm_df_2021[['recency', 'money', 'freq']] = rfm_df_2021[['recency', 'money', 'freq']].astype(float)
# Hiển thị dữ liệu RFM
rfm_df_2021.head()

In [84]:
rfm_df_2021.describe()

In [85]:
rfm_c=rfm_df_2021

In [86]:
rfm_c['r_c'] = pd.qcut(rfm_c['recency'], 5, ['5','4','3','2','1'])
rfm_c['f_c'] = pd.qcut(rfm_c['freq'], 5, ['1','2','3','4','5'])
rfm_c['m_c'] = pd.qcut(rfm_c['money'], 5, ['1','2','3','4','5'])
rfm_c.head()

In [87]:
rfm_c.describe()

In [88]:
rfm_c['RFM_Score'] = rfm_c.r_c.astype(str)+ rfm_c.f_c.astype(str) + rfm_c.m_c.astype(str)
rfm_c.head()

In [89]:
# Specify the file path where you want to save the CSV file
file_path = "./data/rfm_2021.csv"
# Export merged data to CSV
rfm_c.to_csv(file_path, index=False)

In [90]:
# Phân cụm theo recency, frequency và monetary và lưu trữ các cạnh bin
r_bins = pd.qcut(rfm_c['recency'], 5).unique().sort_values()
f_bins = pd.qcut(rfm_c['freq'], 5).unique().sort_values()
m_bins = pd.qcut(rfm_c['money'], 5).unique().sort_values()

# In ra các cạnh bin cho từng chỉ số RFM
print("Recency bin edges:", r_bins)
print("Frequency bin edges:", f_bins)
print("Monetary bin edges:", m_bins)

In [91]:
# Tìm ngày lớn nhất
df_2022['DATE_DIM'] = pd.to_datetime(df_2022['DATE_DIM'])
max_date = df_2022['DATE_DIM'].max()
current_date = max_date + pd.Timedelta(days=1)
print("Ngày hiện tại:", current_date)

In [92]:
# Tính toán Recency bằng cách tính số ngày kể từ ngày gần nhất
df_2022['recency'] = (current_date - df_2022['DATE_DIM']).dt.days
df_2022.head()

In [93]:
# Tính toán Recency bằng cách tính số ngày kể từ ngày gần nhất
df_2022['recency'] = (current_date - df_2022['DATE_DIM']).dt.days
df_2022.head()

In [94]:
# Tính toán Monetary
df_2022['money'] = df_2022['TOTAL_TURNOVER']
df_2022.head()

In [95]:
# Tính toán Frequency bằng cách đếm số lần xuất hiện của mỗi BET_ACCOUNT_NUM_HASH
freq_data = df_2022.groupby('BET_ACCOUNT_NUM_HASH').size().reset_index(name='freq')

In [96]:
# Kết hợp dữ liệu recency, monetary và frequency lại với nhau
rfm_df_2022 = df_2022.groupby('BET_ACCOUNT_NUM_HASH').agg({'recency': 'min', 'money': 'sum'}).reset_index()
rfm_df_2022 = pd.merge(rfm_df_2022, freq_data, on='BET_ACCOUNT_NUM_HASH')
rfm_df_2022[['recency', 'money', 'freq']] = rfm_df_2022[['recency', 'money', 'freq']].astype(float)
# Hiển thị dữ liệu RFM
rfm_df_2022.head()

In [97]:
rfm_df_2022.describe()

In [98]:
df=rfm_df_2022

In [99]:
df['r_c'] = pd.qcut(df['recency'], 4, ['5','4','3','2'])
df['f_c'] = pd.qcut(df['freq'], 5, ['1','2','3','4','5'])
df['m_c'] = pd.qcut(df['money'], 5, ['1','2','3','4','5'])
df.head()

In [100]:
df['RFM_Score'] = df.r_c.astype(str)+ df.f_c.astype(str) + df.m_c.astype(str)
df.head()

In [101]:
# Specify the file path where you want to save the CSV file
file_path = "./data/rfm_2022.csv"
# Export merged data to CSV
df.to_csv(file_path, index=False)

In [102]:
# Phân cụm theo recency, frequency và monetary và lưu trữ các cạnh bin
r_bins = pd.qcut(df['recency'], 4).unique().sort_values()
f_bins = pd.qcut(df['freq'], 5).unique().sort_values()
m_bins = pd.qcut(df['money'], 5).unique().sort_values()

# In ra các cạnh bin cho từng chỉ số RFM
print("Recency bin edges:", r_bins)
print("Frequency bin edges:", f_bins)
print("Monetary bin edges:", m_bins)